# Test LSTM Vocal Confidence Model

Loads the saved LSTM confidence model, Drive scaler files, extracts the same 42 audio features, and predicts a confidence score from one audio or video file.

In [1]:
# Install requirements if running in Colab
!pip install librosa soundfile tensorflow numpy --quiet

In [2]:
# Mount Google Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    print('Not running in Colab, skipping Drive mount.')

Mounted at /content/drive
Google Drive mounted.


In [3]:
from pathlib import Path

import librosa
import numpy as np
import tensorflow as tf

SR = 16000
N_MFCC = 13
MAX_FRAMES = 200
N_FEATURES = 42

print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

GPU available: True


In [4]:
# Update INPUT_PATH to the audio/video file you want to test.
INPUT_PATH = '/content/Test_confidence_aud1.wav'

MODEL_PATH = '/content/drive/MyDrive/PersonaPath/checkpoints/lstm_confidence_final.keras'
SCALER_DIR = '/content/drive/MyDrive/PersonaPath/lstm_features'

INPUT_PATH = Path(INPUT_PATH)
MODEL_PATH = Path(MODEL_PATH)
SCALER_DIR = Path(SCALER_DIR)

print(f'Input: {INPUT_PATH}')
print(f'Model: {MODEL_PATH}')
print(f'Scaler dir: {SCALER_DIR}')

Input: /content/Test_confidence_aud1.wav
Model: /content/drive/MyDrive/PersonaPath/checkpoints/lstm_confidence_final.keras
Scaler dir: /content/drive/MyDrive/PersonaPath/lstm_features


In [6]:
def extract_features(path, sr=SR, n_mfcc=N_MFCC, max_frames=MAX_FRAMES):
    y, sr = librosa.load(str(path), sr=sr)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    pitch = librosa.yin(y, fmin=50, fmax=500)[np.newaxis, :]
    pitch = np.nan_to_num(pitch, nan=0.0)
    rms = librosa.feature.rms(y=y)
    zcr = librosa.feature.zero_crossing_rate(y)

    t = mfcc.shape[1]
    features = np.vstack([
        mfcc,
        delta,
        delta2,
        pitch[:, :t],
        rms[:, :t],
        zcr[:, :t],
    ]).T

    if features.shape[0] >= max_frames:
        features = features[:max_frames]
    else:
        pad = np.zeros((max_frames - features.shape[0], features.shape[1]))
        features = np.vstack([features, pad])

    if features.shape != (MAX_FRAMES, N_FEATURES):
        raise ValueError(f'Expected features shape {(MAX_FRAMES, N_FEATURES)}, got {features.shape}')

    return features.astype(np.float32)


def predict_confidence(input_path, model_path, scaler_dir):
    mean_path = scaler_dir / 'scaler_mean.npy'
    scale_path = scaler_dir / 'scaler_scale.npy'

    if not input_path.exists():
        raise FileNotFoundError(f'Input file not found: {input_path}')
    if not model_path.exists():
        raise FileNotFoundError(f'Model file not found: {model_path}')
    if not mean_path.exists() or not scale_path.exists():
        raise FileNotFoundError(f'Scaler files not found in: {scaler_dir}')

    model = tf.keras.models.load_model(model_path)
    scaler_mean = np.load(mean_path)
    scaler_scale = np.load(scale_path)

    features = extract_features(input_path)
    features = ((features - scaler_mean) / scaler_scale).astype(np.float32)
    features = np.expand_dims(features, axis=0)

    score = float(model.predict(features, verbose=0)[0][0])
    return round(score * 100, 1)

In [7]:
confidence_score = predict_confidence(INPUT_PATH, MODEL_PATH, SCALER_DIR)
print(f'Predicted vocal confidence: {confidence_score}/100')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Predicted vocal confidence: 83.9/100


In [11]:
# Update INPUT_PATH to the audio/video file you want to test.
INPUT_PATH = '/content/test_confidence_aud2.wav'

MODEL_PATH = '/content/drive/MyDrive/PersonaPath/checkpoints/lstm_confidence_final.keras'
SCALER_DIR = '/content/drive/MyDrive/PersonaPath/lstm_features'

INPUT_PATH = Path(INPUT_PATH)
MODEL_PATH = Path(MODEL_PATH)
SCALER_DIR = Path(SCALER_DIR)

print(f'Input: {INPUT_PATH}')
print(f'Model: {MODEL_PATH}')
print(f'Scaler dir: {SCALER_DIR}')

Input: /content/test_confidence_aud2.wav
Model: /content/drive/MyDrive/PersonaPath/checkpoints/lstm_confidence_final.keras
Scaler dir: /content/drive/MyDrive/PersonaPath/lstm_features


In [12]:
def extract_features(path, sr=SR, n_mfcc=N_MFCC, max_frames=MAX_FRAMES):
    y, sr = librosa.load(str(path), sr=sr)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    pitch = librosa.yin(y, fmin=50, fmax=500)[np.newaxis, :]
    pitch = np.nan_to_num(pitch, nan=0.0)
    rms = librosa.feature.rms(y=y)
    zcr = librosa.feature.zero_crossing_rate(y)

    t = mfcc.shape[1]
    features = np.vstack([
        mfcc,
        delta,
        delta2,
        pitch[:, :t],
        rms[:, :t],
        zcr[:, :t],
    ]).T

    if features.shape[0] >= max_frames:
        features = features[:max_frames]
    else:
        pad = np.zeros((max_frames - features.shape[0], features.shape[1]))
        features = np.vstack([features, pad])

    if features.shape != (MAX_FRAMES, N_FEATURES):
        raise ValueError(f'Expected features shape {(MAX_FRAMES, N_FEATURES)}, got {features.shape}')

    return features.astype(np.float32)


def predict_confidence(input_path, model_path, scaler_dir):
    mean_path = scaler_dir / 'scaler_mean.npy'
    scale_path = scaler_dir / 'scaler_scale.npy'

    if not input_path.exists():
        raise FileNotFoundError(f'Input file not found: {input_path}')
    if not model_path.exists():
        raise FileNotFoundError(f'Model file not found: {model_path}')
    if not mean_path.exists() or not scale_path.exists():
        raise FileNotFoundError(f'Scaler files not found in: {scaler_dir}')

    model = tf.keras.models.load_model(model_path)
    scaler_mean = np.load(mean_path)
    scaler_scale = np.load(scale_path)

    features = extract_features(input_path)
    features = ((features - scaler_mean) / scaler_scale).astype(np.float32)
    features = np.expand_dims(features, axis=0)

    score = float(model.predict(features, verbose=0)[0][0])
    return round(score * 100, 1)

In [13]:
confidence_score = predict_confidence(INPUT_PATH, MODEL_PATH, SCALER_DIR)
print(f'Predicted vocal confidence: {confidence_score}/100')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Predicted vocal confidence: 76.5/100


In [15]:
def get_audio_stats(path, sr=16000):
    y, sr = librosa.load(str(path), sr=sr)

    duration = librosa.get_duration(y=y, sr=sr)
    intervals = librosa.effects.split(y, top_db=30)
    voiced_secs = sum((end - start) for start, end in intervals) / sr

    silence_ratio = 1 - (voiced_secs / duration) if duration > 0 else 1
    rms_mean = float(np.mean(librosa.feature.rms(y=y)))

    f0 = librosa.yin(y, fmin=50, fmax=500)
    f0_voiced = f0[f0 > 0]
    pitch_std = float(np.std(f0_voiced)) if len(f0_voiced) > 10 else 0.0

    return {
        "duration": duration,
        "silence_ratio": silence_ratio,
        "rms_mean": rms_mean,
        "pitch_std": pitch_std,
    }


def adjust_confidence_score(model_score, stats):
    adjusted_score = model_score

    if stats["silence_ratio"] > 0.35:
        adjusted_score -= 10

    if stats["rms_mean"] < 0.02:
        adjusted_score -= 8

    if stats["pitch_std"] < 20:
        adjusted_score -= 7

    return round(max(0, min(100, adjusted_score)), 1)


stats = get_audio_stats(INPUT_PATH)
adjusted_score = adjust_confidence_score(confidence_score, stats)

print("Audio stats:")
for key, value in stats.items():
    print(f"{key}: {value:.4f}")

print(f"\nRaw LSTM confidence: {confidence_score}/100")
print(f"Adjusted vocal confidence: {adjusted_score}/100")


Audio stats:
duration: 13.0400
silence_ratio: 0.5755
rms_mean: 0.0302
pitch_std: 119.6248

Raw LSTM confidence: 76.5/100
Adjusted vocal confidence: 66.5/100
